# [SOLUTION] Udaplay Project

## Part 01 - Offline RAG (Retrieval-Augmented Generation)

In this part, you'll build a Vector Database using ChromaDB to store and retrieve video game information efficiently.

**Dataset:** 15 game JSON files inside `project/starter/games/`.

---
### Setup

In [ ]:
# Only needed for Udacity workspace — workaround for pysqlite3
import importlib.util
import sys

if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

#### Create `.env` File

Make sure you have a `config.env` file in this directory with:
```
OPENAI_API_KEY="sk-..."
TAVILY_API_KEY="tvly-..."
```

In [ ]:
# Load environment variables from config.env
load_dotenv("config.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")

print(f"OPENAI_API_KEY set: {OPENAI_API_KEY is not None}")
print(f"TAVILY_API_KEY set: {TAVILY_API_KEY is not None}")

---
### ChromaDB Persistent Client

Create a persistent ChromaDB client so the database is saved to disk and can be reloaded in Part 2.

In [ ]:
# Create (or reload) a persistent ChromaDB client
CHROMA_DB_PATH = "chromadb_udaplay"
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
print(f"ChromaDB persistent client created at: {CHROMA_DB_PATH}")

---
### Embedding Function

Create an OpenAI embedding function to convert text into vector embeddings.

In [ ]:
# Create OpenAI embedding function
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    api_base=OPENAI_BASE_URL
)
print(f"Embedding function created: {type(embedding_fn).__name__}")

---
### Create Collection

Create (or get) the `udaplay` collection in ChromaDB with the embedding function.

In [ ]:
# Create or get the collection (handles both first-run and re-run)
COLLECTION_NAME = "udaplay"

try:
    # Try to get existing collection first
    collection = chroma_client.get_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_fn
    )
    print(f"Reusing existing collection '{COLLECTION_NAME}' ({collection.count()} documents)")
except Exception:
    # Create new collection if it doesn't exist
    collection = chroma_client.create_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_fn
    )
    print(f"Created new collection '{COLLECTION_NAME}'")

---
### Load Game Data & Add to Collection

Read each game JSON file, format the content, and add it to the ChromaDB collection.

In [ ]:
# Directory containing game JSON files
data_dir = "games"

games_loaded = 0

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # Format the content for vector indexing — includes all key game info
    content = (
        f"{game['Name']} is a {game['Genre']} game released for {game['Platform']} "
        f"in {game['YearOfRelease']}. Published by {game['Publisher']}. "
        f"{game['Description']}"
    )

    # Use the file name (e.g. "001") as the document ID
    doc_id = os.path.splitext(file_name)[0]

    # Add to ChromaDB collection
    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )
    games_loaded += 1
    
print(f"Successfully loaded {games_loaded} games into collection '{COLLECTION_NAME}'!")

---
### Test Semantic Search

Let's verify that the vector database works correctly by running some sample queries.

In [ ]:
def search_games(query: str, n_results: int = 3):
    """Helper function to query the vector database."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=['documents', 'distances', 'metadatas']
    )
    return results


def print_search_results(query: str, results):
    """Pretty-print search results."""
    print(f"Query: \"{query}\"\n")
    documents = results['documents'][0]
    distances = results['distances'][0]
    metadatas = results['metadatas'][0]
    
    for i, (doc, dist, meta) in enumerate(zip(documents, distances, metadatas)):
        similarity = 1 - dist  # Convert distance to similarity
        print(f"Result #{i+1} (similarity: {similarity:.4f})")
        print(f"  Name: {meta['Name']}")
        print(f"  Platform: {meta['Platform']}")
        print(f"  Year: {meta['YearOfRelease']}")
        print(f"  Publisher: {meta['Publisher']}")
        print(f"  Genre: {meta['Genre']}")
        print()

In [ ]:
# Test query 1: Find Pokémon games
results_1 = search_games("Pokémon games on Nintendo handhelds")
print_search_results("Pokémon games on Nintendo handhelds", results_1)

In [ ]:
# Test query 2: Find racing games
results_2 = search_games("racing simulation games")
print_search_results("racing simulation games", results_2)

In [ ]:
# Test query 3: Find Spider-Man games
results_3 = search_games("Spider-Man superhero game")
print_search_results("Spider-Man superhero game", results_3)

---
### Summary

The RAG pipeline is now complete:
1. ✅ **Persistent ChromaDB client** created at `chromadb_udaplay/`
2. ✅ **OpenAI embedding function** configured
3. ✅ **`udaplay` collection** created with the embedding function
4. ✅ **15 game documents** loaded and embedded
5. ✅ **Semantic search** works — finding relevant games by natural language queries

The vector database is ready to be used in **Part 2** for the AI agent.